In [9]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('ml2021spring-hw2',output_dir='./ml2021spring-hw2')

print("Path to competition files:", path)

FileExistsError: output_dir is not empty: ./ml2021spring-hw2. Set force_download=True to replace it.

In [17]:
# 加载数据
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim

print("Path to competition files:", path)
file_dir = path + '/timit_11/timit_11'

train_features = np.load(file_dir + '/train_11.npy')
train_labels = np.load(file_dir + '/train_label_11.npy')

test_features = np.load(file_dir + '/test_11.npy')

print(f'train_features.shape: {train_features.shape}')
print(f'train_labels.shape: {train_labels.shape}')
print(f'test_features.shape: {test_features.shape}')


Path to competition files: ./ml2021spring-hw2
train_features.shape: (1229932, 429)
train_labels.shape: (1229932,)
test_features.shape: (451552, 429)


In [16]:
VAL_RATIO = 0.2

percent = int(train_features.shape[0] * (1 - VAL_RATIO))
train_x, train_y, val_x, val_y = train_features[:percent], train_labels[:percent], train_features[percent:], train_labels[percent:]
print('Size of training set: {}'.format(train_x.shape))
print('Size of validation set: {}'.format(val_x.shape))

feat_dim = train_x.shape[1]
print(f'Feature dimension: {feat_dim}')


Size of training set: (983945, 429)
Size of validation set: (245987, 429)
Feature dimension: 429


In [14]:
class TIMITDataset(Dataset):
    def __init__(self, X, y=None):
        self.data = torch.tensor(X, dtype=torch.float32)
        if y is not None:
            y = y.astype(np.long)
            y = torch.tensor(y, dtype=torch.long)
            self.label = y
        else:
            self.label = None

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        if self.label is not None:
            return self.data[idx], self.label[idx]
        else:
            return self.data[idx]



In [15]:
BATCH_SIZE = 64
train_set = TIMITDataset(train_x, train_y)
val_set = TIMITDataset(val_x, val_y)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True) #only shuffle the training data
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)



In [18]:
class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Classifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def get_criterion():
    return nn.CrossEntropyLoss()
    
def get_optimizer(model, lr, weight_decay):
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)